<a href="https://colab.research.google.com/github/Luyao-Xu/3-Class-Speech-Command-Classification/blob/main/03_DL_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

base_path = '/content/drive/MyDrive/speech_commands_project'
log_path = '/content/drive/MyDrive/speech_commands_project/logs'
results_path = '/content/drive/MyDrive/speech_commands_project/results'

os.makedirs(log_path, exist_ok=True)
os.makedirs(results_path, exist_ok=True)

print(f"Directories ready.\nLogs: {log_path}\nResults: {results_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directories ready.
Logs: /content/drive/MyDrive/speech_commands_project/logs
Results: /content/drive/MyDrive/speech_commands_project/results


### Data Preparation (The 2D Loader)


In [ ]:
!pip install "numpy<2.0"
!pip install -q tf_keras
!pip install -q tensorflow-model-optimization
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import numpy as np
import tensorflow as tf
import tf_keras
from tf_keras import layers, models
from tf_keras.optimizers import Adam
import tensorflow_model_optimization as tfmot


In [ ]:
# --- EXPERIMENT CONFIGURATION LEDGER ---
# Experiment ID: EXP_2026_05_08_01
# Description: Comparison of Standard CNN, SqueezeNet, and UltraLight with QAT/PTQ
# Author: [Xu Luyao]

config = {
    "version": "1.2.0",
    "input_shape": (40, 32, 1),
    "num_classes": 3,
    "batch_size": 32,
    "initial_epochs": 30,
    "qat_fine_tune_epochs": 8,
    "learning_rate_base": 0.001,
    "learning_rate_qat": 1e-5,
    "random_seed": 42
}

import numpy as np
import tensorflow as tf
tf.random.set_seed(config["random_seed"])
np.random.seed(config["random_seed"])

In [ ]:
# Path to your processed data
processed_path = '/content/drive/MyDrive/speech_commands_project/data/processed'

# Load files
X_train = np.load(os.path.join(processed_path, 'X_train.npy'))
y_train = np.load(os.path.join(processed_path, 'y_train.npy'))
X_val = np.load(os.path.join(processed_path, 'X_val.npy'))
y_val = np.load(os.path.join(processed_path, 'y_val.npy'))
X_test = np.load(os.path.join(processed_path, 'X_test.npy'))
y_test = np.load(os.path.join(processed_path, 'y_test.npy'))

# Reshape for CNN: (Samples, Height, Width, Channels)
# Your MFCCs are 40x32
X_train = X_train.reshape(X_train.shape[0], 40, 32, 1)
X_val = X_val.reshape(X_val.shape[0], 40, 32, 1)
X_test = X_test.reshape(X_test.shape[0], 40, 32, 1)

print(f"Data ready for CNN. Train shape: {X_train.shape}")

Data ready for CNN. Train shape: (7492, 40, 32, 1)


#### Model A:Simple 2D CNN (Baseline)
##### Uses classic 2D convolutions. This is the high-parameter reference point to show why optimization is necessary.

In [ ]:
def build_simple_cnn_for_qat(input_shape, num_classes):
    return tf_keras.Sequential([
        tf_keras.layers.Input(shape=input_shape),
        tf_keras.layers.Conv2D(32, (3,3), activation='relu'),
        tf_keras.layers.MaxPooling2D((2,2)),
        tf_keras.layers.Conv2D(64, (3,3), activation='relu'),
        tf_keras.layers.MaxPooling2D((2,2)),
        tf_keras.layers.Flatten(),
        tf_keras.layers.Dense(64, activation='relu'),
        tf_keras.layers.Dropout(0.3),
        tf_keras.layers.Dense(3, activation='softmax')
    ])

model_a = build_simple_cnn_for_qat((40, 32, 1), 3)
model_a.compile(
    optimizer=tf_keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_a.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_17 (Conv2D)          (None, 38, 30, 32)        320       
                                                                 
 max_pooling2d_5 (MaxPoolin  (None, 19, 15, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_18 (Conv2D)          (None, 17, 13, 64)        18496     
                                                                 
 max_pooling2d_6 (MaxPoolin  (None, 8, 6, 64)          0         
 g2D)                                                            
                                                                 
 flatten_1 (Flatten)         (None, 3072)              0         
                                                                 
 dense_5 (Dense)             (None, 64)               

### Model B:Mini-SqueezeNet
##### A specific lightweight architecture, uses Fire Modules (Squeeze & Expand).
 how 1*1 convolutions can 'squeeze' information to save memory.

In [ ]:
def fire_module(x, squeeze, expand):
    s = layers.Conv2D(squeeze, (1, 1), padding='same', activation='relu')(x)
    e1 = layers.Conv2D(expand, (1, 1), padding='same', activation='relu')(s)
    e3 = layers.Conv2D(expand, (3, 3), padding='same', activation='relu')(s)
    return layers.Concatenate()([e1, e3])

def build_squeezenet(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(16, (3, 3), strides=(1, 1), padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # The Fire Modules
    x = fire_module(x, squeeze=8, expand=16)
    x = fire_module(x, squeeze=8, expand=16)

    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs, name="Model_B_SqueezeNet")
    return model

# Create and inspect
model_b = build_squeezenet((40, 32, 1), 3)
model_b.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_b.summary()

Model: "Model_B_SqueezeNet"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_5 (InputLayer)        [(None, 40, 32, 1)]          0         []                            
                                                                                                  
 conv2d_19 (Conv2D)          (None, 40, 32, 16)           160       ['input_5[0][0]']             
                                                                                                  
 max_pooling2d_7 (MaxPoolin  (None, 20, 16, 16)           0         ['conv2d_19[0][0]']           
 g2D)                                                                                             
                                                                                                  
 conv2d_20 (Conv2D)          (None, 20, 16, 8)            136       ['max_pooling

### Model C:Ultra-Lightweight (MobileNet-style)
##### MobileNet-inspired, uses Depthwise Separable Convolutions and Global Average Pooling.

In [ ]:
def build_ultralight_cnn(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Block 1: Feature Extraction
        layers.SeparableConv2D(16, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Block 2: Feature Extraction
        layers.SeparableConv2D(32, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # The "Embedded Trick": Global Average Pooling instead of Flatten
        layers.GlobalAveragePooling2D(),

        # Final Classification Head
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# 1. Initialize Model C
model_c = build_ultralight_cnn((40, 32, 1), 3)

# 2. Compile it (This ensures the 'Adam' name is recognized)
model_c.compile(optimizer=Adam(learning_rate=0.001),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])

# 3. View Summary
model_c.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 separable_conv2d_2 (Separa  (None, 40, 32, 16)        41        
 bleConv2D)                                                      
                                                                 
 max_pooling2d_8 (MaxPoolin  (None, 20, 16, 16)        0         
 g2D)                                                            
                                                                 
 separable_conv2d_3 (Separa  (None, 20, 16, 32)        688       
 bleConv2D)                                                      
                                                                 
 max_pooling2d_9 (MaxPoolin  (None, 10, 8, 32)         0         
 g2D)                                                            
                                                                 
 global_average_pooling2d_3  (None, 32)               

##**Traing**
##### I utilized a batch size of 32 and 30 epochs to ensure stable gradient convergence while avoiding overfitting on the phonetically similar classes. I maintained the standard Adam learning rate of 0.001 for the initial training phase.

In [ ]:
models_to_train = {
    "Model_A_Standard": model_a,
    "Model_B_Separable": model_b,
    "Model_C_UltraLight": model_c
}

EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 0.001

histories = {}

for name, model in models_to_train.items():
    print(f"\n Starting Training for {name} ")

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val),
        verbose=1
    )
    histories[name] = history

    history_df = pd.DataFrame(history.history)
    history_csv = os.path.join(log_path, f"{name}_training_log.csv")
    history_df.to_csv(history_csv, index=False)

    print(f" {name} log successfully saved to: {history_csv}")
    print(f"{name} Training Complete.")



 Starting Training for Model_A_Standard 
Epoch 1/30
235/235 [==============================] - 15s 60ms/step - loss: 0.7449 - accuracy: 0.7304 - val_loss: 0.3859 - val_accuracy: 0.8356
Epoch 2/30
235/235 [==============================] - 12s 50ms/step - loss: 0.3436 - accuracy: 0.8576 - val_loss: 0.2613 - val_accuracy: 0.9018
Epoch 3/30
235/235 [==============================] - 12s 50ms/step - loss: 0.2522 - accuracy: 0.9052 - val_loss: 0.2697 - val_accuracy: 0.8901
Epoch 4/30
235/235 [==============================] - 10s 42ms/step - loss: 0.2020 - accuracy: 0.9204 - val_loss: 0.2102 - val_accuracy: 0.9232
Epoch 5/30
235/235 [==============================] - 12s 50ms/step - loss: 0.1756 - accuracy: 0.9331 - val_loss: 0.2119 - val_accuracy: 0.9221
Epoch 6/30
235/235 [==============================] - 12s 50ms/step - loss: 0.1487 - accuracy: 0.9426 - val_loss: 0.2040 - val_accuracy: 0.9178
Epoch 7/30
235/235 [==============================] - 12s 50ms/step - loss: 0.1417 - accuracy:

### The Quantitative Comparison

In [ ]:
import pandas as pd
def generate_comparison_table(models_dict, history_dict):
    stats = []
    for name, model in models_dict.items():
        params = model.count_params()
        # Memory estimation: each param is 4 bytes (float32)
        size_kb = (params * 4) / 1024

        # Get final validation accuracy
        final_val_acc = history_dict[name].history['val_accuracy'][-1]

        stats.append({
            "Model Architecture": name,
            "Total Parameters": f"{params:,}",
            "Est. Memory (KB)": f"{size_kb:.2f}",
            "Final Accuracy": f"{final_val_acc:.2%}"
        })
    return pd.DataFrame(stats)

# Generate and print
results_df = generate_comparison_table(models_to_train, histories)
print("\n Final Project Analysis: DL Models")
print(results_df)


 Final Project Analysis: DL Models
   Model Architecture Total Parameters Est. Memory (KB) Final Accuracy
0    Model_A_Standard          215,683           842.51         92.10%
1   Model_B_Separable            3,283            12.82         91.57%
2  Model_C_UltraLight            1,884             7.36         83.24%


## Post-Training Quantization (PTQ)
##### During Post-Training Quantization (PTQ), a warning regarding missing input/output statistics was noted. While the internal weights were successfully quantized to INT8 (achieving the desired memory reduction), the model interface remained in Float32 format. For deployment on a strict integer-only hardware target, further calibration of the input/output tensors would be required

In [ ]:
ptq_conversion_stats = []

def representative_data_gen():
    for i in range(100):
        data = np.expand_dims(X_train[i], axis=0).astype(np.float32)
        yield [data]

for name, model in models_to_train.items():
    print(f"\n Processing: {name}")

    try:
        # 1. Export as SavedModel (Required for stable TFLite conversion)
        saved_model_path = os.path.join(results_path, f'{name}_ptq_saved')
        model.export(saved_model_path)

        # 2. Conversion Configuration
        converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_data_gen
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.float32
        converter.inference_output_type = tf.float32

        tflite_model = converter.convert()

        file_path = os.path.join(results_path, f'{name}_ptq.tflite')
        with open(file_path, 'wb') as f:
            f.write(tflite_model)

        # 4. CAPTURE DATA: This is where we populate the ledger
        size_kb = len(tflite_model) / 1024
        ptq_conversion_stats.append({
            "Model_Name": name,
            "Format": "TFLite_PTQ",
            "Size_KB": round(size_kb, 2),
            "File_Path": file_path,
            "Timestamp": pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')
        })

        # 5. Archive training history if available
        if 'histories' in globals() and name in histories:
            history_df = pd.DataFrame(histories[name].history)
            history_csv = os.path.join(log_path, f"{name}_ptq_log.csv")
            history_df.to_csv(history_csv, index=False)

        print(f" Success: {name} | Size: {size_kb:.2f} KB")

    except Exception as e:
        print(f" Error converting {name}: {str(e)}")

# --- STEP 3: Save and Verify Master Ledger ---
if ptq_conversion_stats:
    ptq_results_df = pd.DataFrame(ptq_conversion_stats)
    master_csv_path = os.path.join(results_path, 'ptq_conversion_master_ledger.csv')
    ptq_results_df.to_csv(master_csv_path, index=False)

    print(f"\n MASTER LEDGER UPDATED: {master_csv_path}")
    print(ptq_results_df)
else:
    print("\n ALERT: No conversion data was captured. Please check the error messages above.")


 Processing: Model_A_Standard
Saved artifact at '/content/drive/MyDrive/speech_commands_project/results/Model_A_Standard_ptq_saved'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40, 32, 1), dtype=tf.float32, name='input_4')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  132293192434320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132292813207504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132293192443728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132292747973776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132292747973584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132292747975504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132292747973968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132292747974544: TensorSpec(shape=(), dtype=tf.resource, name=None)
 Success: Model_A_Standard | Size: 218.98 KB

 Processing: Model_B_

In [ ]:
def evaluate_tflite_accuracy(file_path, X_test, y_test):
    # 1. Initialize the interpreter
    interpreter = tf.lite.Interpreter(model_path=file_path)
    interpreter.allocate_tensors()

    # 2. Get input and output details
    input_index = interpreter.get_input_details()[0]["index"]
    output_index = interpreter.get_output_details()[0]["index"]

    prediction_count = 0

    # 3. Inference loop
    for i in range(len(X_test)):
        # Add batch dimension and ensure float32
        input_data = np.expand_dims(X_test[i], axis=0).astype(np.float32)

        interpreter.set_tensor(input_index, input_data)
        interpreter.invoke()

        output_data = interpreter.get_tensor(output_index)
        prediction = np.argmax(output_data)

        if prediction == y_test[i]:
            prediction_count += 1

    accuracy = prediction_count / len(X_test)
    return accuracy

# --- Run the Evaluation for all PTQ Models ---
ptq_results = []

for name, path in tflite_ptq_files.items():
    print(f"Evaluating PTQ Accuracy for {name}...")
    acc = evaluate_tflite_accuracy(path, X_test, y_test)

    # Traceability: logging size and accuracy together
    size_kb = os.path.getsize(path) / 1024
    ptq_results.append({
        "Model": name,
        "PTQ Accuracy": f"{acc:.2%}",
        "PTQ Size (KB)": f"{size_kb:.2f}"
    })
# Convert to DataFrame for structured logging
df_ptq = pd.DataFrame(ptq_results)

final_results_csv = os.path.join(results_path, 'ptq_final_evaluation_results.csv')
df_ptq.to_csv(final_results_csv, index=False)

print("\n--- PTQ Experimental Results ---")
print(df_ptq)

Evaluating PTQ Accuracy for Model_A_Standard...


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Evaluating PTQ Accuracy for Model_B_Separable...
Evaluating PTQ Accuracy for Model_C_UltraLight...

--- PTQ Experimental Results ---
                Model PTQ Accuracy PTQ Size (KB)
0    Model_A_Standard       92.42%        218.98
1   Model_B_Separable       92.64%         12.22
2  Model_C_UltraLight       83.14%          9.65


## **Quantization-Aware Training (QAT)**

In [ ]:
# --- 1. Wrap Model B and C for QAT ---
print("Applying QAT wrappers to Model A B and C...")
model_a_qat = tfmot.quantization.keras.quantize_model(model_a)
model_b_qat = tfmot.quantization.keras.quantize_model(model_b)
model_c_qat = tfmot.quantization.keras.quantize_model(model_c)

# --- 2. Compile QAT Versions ---
# We use a lower learning rate (1e-5) for QAT fine-tuning to preserve features
qat_models = {
    "Model_A_Standard": model_a_qat,
    "Model_B_Separable": model_b_qat,
    "Model_C_UltraLight": model_c_qat
}

for name, q_model in qat_models.items():
    q_model.compile(
        optimizer=tf_keras.optimizers.Adam(learning_rate=1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = q_model.fit(
        X_train, y_train,
        epochs=8,
        batch_size=32,
        validation_data=(X_val, y_val),
        verbose=1
    )

    history_df = pd.DataFrame(histories[name].history)
    history_csv = os.path.join(log_path, f"{name}_qat_log.csv")
    history_df.to_csv(history_csv, index=False)

    print(f"{name} log saved to {history_csv}")
    print(f"\n Fine-tuning QAT for {name}...")

Applying QAT wrappers to Model A B and C...
Epoch 1/8
235/235 [==============================] - 16s 59ms/step - loss: 0.2063 - accuracy: 0.9234 - val_loss: 0.4551 - val_accuracy: 0.8794
Epoch 2/8
235/235 [==============================] - 13s 57ms/step - loss: 0.0420 - accuracy: 0.9847 - val_loss: 0.3958 - val_accuracy: 0.9146
Epoch 3/8
235/235 [==============================] - 14s 58ms/step - loss: 0.0261 - accuracy: 0.9911 - val_loss: 0.3926 - val_accuracy: 0.9242
Epoch 4/8
235/235 [==============================] - 13s 57ms/step - loss: 0.0190 - accuracy: 0.9935 - val_loss: 0.3930 - val_accuracy: 0.9264
Epoch 5/8
235/235 [==============================] - 13s 57ms/step - loss: 0.0156 - accuracy: 0.9947 - val_loss: 0.3916 - val_accuracy: 0.9253
Epoch 6/8
235/235 [==============================] - 16s 69ms/step - loss: 0.0134 - accuracy: 0.9961 - val_loss: 0.3979 - val_accuracy: 0.9264
Epoch 7/8
235/235 [==============================] - 13s 57ms/step - loss: 0.0122 - accuracy: 0.99

#### Convert QAT Models to TFLite

In [ ]:
qat_tflite_files = {}

# Dictionary of trained QAT models
final_qat_models = {
    "Model_A_Standard": model_a_qat,
    "Model_B_Separable": model_b_qat,
    "Model_C_UltraLight": model_c_qat
}

for name, model in final_qat_models.items():
    print(f"Converting {name} QAT to TFLite...")

    # Using export/saved_model to ensure consistency with your PTQ cell
    saved_qat_path = os.path.join(results_path, f'{name}_qat_saved')
    model.export(saved_qat_path)

    converter = tf.lite.TFLiteConverter.from_saved_model(saved_qat_path)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    tflite_model = converter.convert()

    file_path = os.path.join(results_path, f'{name}_qat.tflite')
    with open(file_path, 'wb') as f:
        f.write(tflite_model)

    qat_tflite_files[name] = file_path
    size_kb = len(tflite_model) / 1024
    print(f"Saved to {file_path} ({len(tflite_model)/1024:.2f} KB)")
    if name in qat_models:
      history_df = pd.DataFrame(history.history)
      history_csv = os.path.join(log_path, f"{name}_qat_training_log.csv")
      history_df.to_csv(history_csv, index=False)

      print(f"QAT Fine-tune log saved to {history_csv}")

Converting Model_A_Standard QAT to TFLite...
Saved artifact at '/content/drive/MyDrive/speech_commands_project/results/Model_A_Standard_qat_saved'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40, 32, 1), dtype=tf.float32, name='input_4')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  132292702090832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132292702084688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132293188245008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132292702077968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132292702077584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132292702092752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132292702078736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132292702079312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132292702086992: TensorSpec(shape=(), dtype=tf.reso

In [ ]:
import time
def evaluate_tflite_and_measure(file_path, X_test, y_test):
    # Initialize Interpreter
    interpreter = tf.lite.Interpreter(model_path=file_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    correct = 0
    start_time = time.time()

    # Run inference on test set
    for i in range(len(X_test)):
        input_data = X_test[i:i+1].astype(np.float32)
        interpreter.set_tensor(input_details['index'], input_data)
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_details['index'])

        if np.argmax(output_data) == y_test[i]:
            correct += 1

    # Calculate Metrics
    latency = (time.time() - start_time) / len(X_test) * 1000 # ms per sample
    accuracy = correct / len(X_test)
    size_kb = os.path.getsize(file_path) / 1024

    return accuracy, size_kb, latency

# --- Generate the Final Comparison Table ---
benchmark_results = []

# Evaluate PTQ Models (from your previous cell)
for name, path in tflite_ptq_files.items():
    acc, size, lat = evaluate_tflite_and_measure(path, X_test, y_test)
    benchmark_results.append({"Model": name, "Type": "PTQ", "Acc": f"{acc:.2%}", "Size (KB)": f"{size:.1f}", "Latency (ms)": f"{lat:.2f}"})

# Evaluate QAT Models
for name, path in qat_tflite_files.items():
    acc, size, lat = evaluate_tflite_and_measure(path, X_test, y_test)
    benchmark_results.append({"Model": name, "Type": "QAT", "Acc": f"{acc:.2%}", "Size (KB)": f"{size:.1f}", "Latency (ms)": f"{lat:.2f}"})


results_df = pd.DataFrame(benchmark_results)
master_csv_path = os.path.join(results_path, 'compression_analysis.csv')
results_df.to_csv(master_csv_path, index=False)

print("\n Final Compression Experiment Analysis:")
print(results_df)

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



 Final Compression Experiment Analysis:
                Model Type     Acc Size (KB) Latency (ms)
0    Model_A_Standard  PTQ  92.42%     219.0         0.30
1   Model_B_Separable  PTQ  92.64%      12.2         0.17
2  Model_C_UltraLight  PTQ  83.14%       9.6         0.09
3    Model_A_Standard  QAT  93.28%     217.9         0.32
4   Model_B_Separable  QAT  92.21%      13.3         0.16
5  Model_C_UltraLight  QAT  83.35%       9.5         0.08
